In [1]:
# Clone the repository into Colab workspace
!rm -rf /content/Verifiable-LLM-Unlearning-master
!git clone https://github.com/peiyan0/Verifiable-LLM-Unlearning.git /content/Verifiable-LLM-Unlearning-master

Cloning into '/content/Verifiable-LLM-Unlearning-master'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 141 (delta 58), reused 127 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 686.23 KiB | 12.71 MiB/s, done.
Resolving deltas: 100% (58/58), done.


In [2]:
# !unzip -q -o /content/Verifiable-LLM-Unlearning-master.zip -d /content
!pip install -q --upgrade torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu126
!pip install -q -r /content/Verifiable-LLM-Unlearning-master/requirements.txt
!pip install -q transformers==4.56.0 datasets==3.6.0 peft==0.17.1 accelerate==1.10.0 bitsandbytes==0.46.1 opacus==1.6.0 einops

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
%cd /content/Verifiable-LLM-Unlearning-master

# Reset ledger + old results so compliance is not polluted by stale runs
!rm -rf results/*.json audit_ledger* .ledger* 2>/dev/null || true

# Run the GPU pilot once (do NOT also call experiments/run_all.sh here,
# as it invokes run_pilot_audit.py again)
!TRANSFORMERS_VERBOSITY=info python scripts/run_pilot_audit.py --device cuda

/content/Verifiable-LLM-Unlearning-master
[data] forget=200 holdout=200 retain=400 (real locuslab/TOFU cache)
Loaded existing audit ledger with 19 entries.
JAX version 0.11.1, Flax version 0.11.2 available.
tokenizer_config.json: 100% 237/237 [00:00<00:00, 1.49MB/s]
vocab.json: 798kB [00:00, 80.8MB/s]
merges.txt: 456kB [00:00, 146MB/s]
tokenizer.json: 2.11MB [00:00, 177MB/s]
added_tokens.json: 1.08kB [00:00, 6.29MB/s]
special_tokens_map.json: 100% 99.0/99.0 [00:00<00:00, 641kB/s]
loading file vocab.json from cache at /root/.cache/huggingface/hub/models--microsoft--phi-1.5/snapshots/77aa61eeac94fbf33d492b9f2744c98b42d5b5eb/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--microsoft--phi-1.5/snapshots/77aa61eeac94fbf33d492b9f2744c98b42d5b5eb/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--microsoft--phi-1.5/snapshots/77aa61eeac94fbf33d492b9f2744c98b42d5b5eb/tokenizer.json
loading file added_tokens.json from cac

In [4]:
import json, pathlib

def fmt(val):
    return f"{val:.4f}" if isinstance(val, (int, float)) else str(val)

for name in ["ft_audit.json", "ga_audit.json", "dp2u_audit.json"]:
    p_res = pathlib.Path("results") / name
    if not p_res.exists():
        print(f"MISSING: {name}")
        continue

    d = json.loads(p_res.read_text())

    # Fallback in case metrics are nested under a sub-key like 'metrics' or 'summary'
    metrics = d.get("metrics", d) if isinstance(d.get("metrics"), dict) else d

    auc = metrics.get("auc")
    tpr = metrics.get("tpr_at_1pct")
    ci = metrics.get("auc_ci")
    n_forget = metrics.get("n_forget")
    n_holdout = metrics.get("n_holdout")

    print(f"{name:16s} AUC={fmt(auc)}  TPR@1%={fmt(tpr)}  "
          f"CI={ci}  n_forget={n_forget} n_holdout={n_holdout}")

    # If AUC was None, print available keys to help inspect the JSON schema
    if auc is None:
        print(f"   -> Available keys in {name}: {list(d.keys())}")

cs_path = pathlib.Path("results/compliance_summary.json")
if cs_path.exists():
    print("\n--- compliance ---")
    cs = json.loads(cs_path.read_text())
    for k, v in cs.items():
        print(f"  {k:20s} {v}")
else:
    print("\n--- compliance_summary.json not found (certification was refused) ---")

ft_audit.json    AUC=None  TPR@1%=None  CI=None  n_forget=200 n_holdout=None
   -> Available keys in ft_audit.json: ['generated_by', 'base_model', 'device', 'seed', 'config_epsilon_sweep', 'dataset', 'author_cluster_method', 'adapter', 'role', 'criterion', 'MIA_AUC', 'TPR_at_1pct_FPR', 'n_forget', 'n_nonmember', 'author_clustered_ci', 'record_level_ci_deprecated', 'scoring']
ga_audit.json    AUC=None  TPR@1%=None  CI=None  n_forget=200 n_holdout=None
   -> Available keys in ga_audit.json: ['generated_by', 'base_model', 'device', 'seed', 'config_epsilon_sweep', 'dataset', 'author_cluster_method', 'adapter', 'role', 'ga_record', 'MIA_AUC', 'TPR_at_1pct_FPR', 'n_forget', 'n_nonmember', 'author_clustered_ci', 'record_level_ci_deprecated', 'scoring']
dp2u_audit.json  AUC=None  TPR@1%=None  CI=None  n_forget=200 n_holdout=None
   -> Available keys in dp2u_audit.json: ['generated_by', 'base_model', 'device', 'seed', 'config_epsilon_sweep', 'dataset', 'author_cluster_method', 'adapter', 'role'

In [5]:
from google.colab import userdata
import os

# Retrieve token from Colab Secrets (or replace with "ghp_your_token_here")
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/peiyan0/Verifiable-LLM-Unlearning.git"

# Configure git identity
!git config user.email "colab-runner@users.noreply.github.com"
!git config user.name "Colab Runner"

# Set authenticated remote URL
!git remote set-url origin "{REPO_URL}"

# Stage, commit, and push the results directory
!git add results/ src/audit_ledger.json
!git commit -m "output: add GPU pilot audit results"
!git push origin master

[master 0a8338a] output: add GPU pilot audit results
 6 files changed, 259 insertions(+), 29 deletions(-)
 create mode 100644 results/pilot_certificate.json
 create mode 100644 results/pr1_isolation.json
Enumerating objects: 17, done.
Counting objects: 100% (17/17), done.
Delta compression using up to 2 threads
Compressing objects: 100% (10/10), done.
Writing objects: 100% (10/10), 3.06 KiB | 348.00 KiB/s, done.
Total 10 (delta 7), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (7/7), completed with 6 local objects.
To https://github.com/peiyan0/Verifiable-LLM-Unlearning.git
   0ac6472..0a8338a  master -> master
